In [0]:
# Cell 1: Load orders.csv
orders_df = spark.read.csv(
    "/Workspace/orders.csv",
    header=True,
    inferSchema=True
)
display(orders_df)

order_id,customer_id,product_name,order_date,delivery_date,region,issue
1,101,Laptop,2024-01-01,2024-01-12,North,Weather
2,102,Phone,2024-01-03,2024-01-09,South,null
3,103,Tablet,2024-01-05,2024-01-15,East,Logistics
4,101,Monitor,2024-01-07,2024-01-20,North,Lost
5,104,Keyboard,2024-01-08,2024-01-20,West,Logistics
6,102,Mouse,2024-01-10,2024-01-11,South,null
7,105,Headphones,2024-01-12,2024-01-25,North,Weather
8,106,Webcam,2024-01-13,2024-01-19,East,Logistics
9,107,Speaker,2024-01-14,2024-01-16,West,null
10,108,Charger,2024-01-15,2024-01-28,South,Delay


In [0]:
# Cell 2: Load customers.csv
customers_df = spark.read.csv(
    "/Workspace/customers.csv",
    header=True,
    inferSchema=True
)
display(customers_df)

customer_id,name,email,phone,region
101,Alice Johnson,alice@email.com,9876543210,North
102,Bob Smith,bob@email.com,9123456780,South
103,Carol White,carol@email.com,9988776655,East
104,David Brown,david@email.com,9871234560,West
105,Eva Green,eva@email.com,9765432100,North
106,Frank Black,frank@email.com,9654321890,East
107,Grace Lee,grace@email.com,9543218760,West
108,Henry Adams,henry@email.com,9432187650,South


In [0]:
# Cell 3: Transform orders – calculate delay and flag
from pyspark.sql.functions import datediff, to_date, when, col, sum as spark_sum

orders_df = orders_df \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("delivery_date", to_date(col("delivery_date"), "yyyy-MM-dd")) \
    .withColumn("delay_days", datediff(col("delivery_date"), col("order_date"))) \
    .withColumn("delayed", when(col("delay_days") > 5, 1).otherwise(0))

display(orders_df)

order_id,customer_id,product_name,order_date,delivery_date,region,issue,delay_days,delayed
1,101,Laptop,2024-01-01,2024-01-12,North,Weather,11,1
2,102,Phone,2024-01-03,2024-01-09,South,null,6,1
3,103,Tablet,2024-01-05,2024-01-15,East,Logistics,10,1
4,101,Monitor,2024-01-07,2024-01-20,North,Lost,13,1
5,104,Keyboard,2024-01-08,2024-01-20,West,Logistics,12,1
6,102,Mouse,2024-01-10,2024-01-11,South,null,1,0
7,105,Headphones,2024-01-12,2024-01-25,North,Weather,13,1
8,106,Webcam,2024-01-13,2024-01-19,East,Logistics,6,1
9,107,Speaker,2024-01-14,2024-01-16,West,null,2,0
10,108,Charger,2024-01-15,2024-01-28,South,Delay,13,1


In [0]:
# Cell 4: Update latest delivery status per customer
latest_status = orders_df.groupBy("customer_id").agg(
    {"delay_days": "max", "delayed": "sum"}
).withColumnRenamed("max(delay_days)", "max_delay_days") \
 .withColumnRenamed("sum(delayed)", "total_delayed_orders")

display(latest_status)

customer_id,max_delay_days,total_delayed_orders
105,13,1
104,12,1
108,13,1
106,6,1
103,13,2
107,2,0
102,6,1
101,13,3


In [0]:
# Cell 5: Join orders with customers (drop duplicate region from customers)
joined_df = orders_df.join(
    customers_df.drop("region"),
    on="customer_id",
    how="left"
)
display(joined_df)

customer_id,order_id,product_name,order_date,delivery_date,region,issue,delay_days,delayed,name,email,phone
101,1,Laptop,2024-01-01,2024-01-12,North,Weather,11,1,Alice Johnson,alice@email.com,9876543210
102,2,Phone,2024-01-03,2024-01-09,South,null,6,1,Bob Smith,bob@email.com,9123456780
103,3,Tablet,2024-01-05,2024-01-15,East,Logistics,10,1,Carol White,carol@email.com,9988776655
101,4,Monitor,2024-01-07,2024-01-20,North,Lost,13,1,Alice Johnson,alice@email.com,9876543210
104,5,Keyboard,2024-01-08,2024-01-20,West,Logistics,12,1,David Brown,david@email.com,9871234560
102,6,Mouse,2024-01-10,2024-01-11,South,null,1,0,Bob Smith,bob@email.com,9123456780
105,7,Headphones,2024-01-12,2024-01-25,North,Weather,13,1,Eva Green,eva@email.com,9765432100
106,8,Webcam,2024-01-13,2024-01-19,East,Logistics,6,1,Frank Black,frank@email.com,9654321890
107,9,Speaker,2024-01-14,2024-01-16,West,null,2,0,Grace Lee,grace@email.com,9543218760
108,10,Charger,2024-01-15,2024-01-28,South,Delay,13,1,Henry Adams,henry@email.com,9432187650


In [0]:
# Cell 6: Top 5 Delayed Customers using SQL
orders_df.createOrReplaceTempView("orders")

top5 = spark.sql("""
    SELECT customer_id,
           COUNT(*) AS total_orders,
           SUM(delayed) AS total_delayed_orders,
           MAX(delay_days) AS max_delay_days
    FROM orders
    GROUP BY customer_id
    ORDER BY total_delayed_orders DESC
    LIMIT 5
""")
display(top5)

customer_id,total_orders,total_delayed_orders,max_delay_days
101,3,3,13
103,2,2,13
104,1,1,12
105,1,1,13
108,1,1,13


In [0]:
# Cell 7: Region-wise delay summary
region_summary = joined_df.groupBy("region").agg(
    spark_sum("delayed").alias("delayed_orders")
)
display(region_summary)

region,delayed_orders
North,4
East,3
South,2
West,1


In [0]:
import pandas as pd
region_summary_pd = region_summary.toPandas()
region_summary_pd.to_csv("/Workspace/output_region_summary.csv", index=False)

print("ETL Pipeline Complete!")
print(region_summary_pd)

ETL Pipeline Complete!
  region  delayed_orders
0  North               4
1   East               3
2  South               2
3   West               1
